# PINN Inverse Heat Source (1D) — v6 (weak-form PDE)

**Problem**: recover unknown source $f(x)$ from sparse noisy measurements:
$$-\frac{d^2T}{dx^2} = f(x), \quad x\in(0,1), \quad T(0)=T(1)=0$$

## Why the strong-form approach fails

The strong-form residual $\|T'' + f\|^2$ requires computing $T''$ via autograd.
Even when $T$ is accurate to $10^{-3}$, $T''$ has errors ~0.15 (see v5 results).
Any smoothness penalty over-regularizes $T''$ into a flat function.

## v6: weak-form PDE (only needs $T'$!)

Integration by parts: $\int T'\phi_k'\,dx = \int f\,\phi_k\,dx$ with $\phi_k = \sin(k\pi x)$.

**Advantages**: (1) Only first derivatives (accurate), (2) Integration smooths noise, (3) Each test function probes a frequency.

| Phase | Trains | Loss | Epochs |
|-------|--------|------|--------|
| 1 | $\mathcal{N}_T$ only | Data + BC | 20k |
| 2 | $\mathcal{N}_f$ only | **Weak PDE** + Tikhonov | 50k |
| 3 | Both | Weak PDE + Data + BC (ramp) | 80k + L-BFGS |

In [ ]:
# Fix sympy/torch compatibility (Colab only)
import sympy, subprocess, sys
if int(sympy.__version__.split('.')[1]) >= 14:
    print(f'sympy {sympy.__version__} incompatible')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sympy==1.13.1'])
    print('Fixed. RESTART RUNTIME now, then skip this cell.')
else:
    print(f'sympy {sympy.__version__} OK')

In [ ]:
import sys, os, gc

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('/content/pinn-inverse-heat'):
        !git clone https://github.com/MaximeAuger/pinn-inverse-heat.git
    sys.path.insert(0, '/content/pinn-inverse-heat/src')
    RESULTS_DIR = '/content/results'
else:
    sys.path.insert(0, '../src')
    RESULTS_DIR = '../results'

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results dir: {RESULTS_DIR}')

import torch, torch.nn as nn, torch.optim as optim
import numpy as np, matplotlib.pyplot as plt, matplotlib.animation as animation
torch.manual_seed(42); np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def free_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f'PyTorch {torch.__version__} | {device}')

In [ ]:
# === Architecture: Fourier-featured MLP ===
class FourierFeatures(nn.Module):
    def __init__(self, K=6):
        super().__init__()
        self.K = K
        self.register_buffer('freqs', torch.arange(1, K+1, dtype=torch.float32)*np.pi)
    @property
    def out_dim(self): return 1 + 2*self.K
    def forward(self, x):
        xf = x * self.freqs
        return torch.cat([x, torch.sin(xf), torch.cos(xf)], dim=-1)

class FourierMLP(nn.Module):
    def __init__(self, hidden, K=6, scale=1.0):
        super().__init__()
        self.ff = FourierFeatures(K)
        dims = [self.ff.out_dim] + list(hidden) + [1]
        net = []
        for i in range(len(dims)-1):
            lin = nn.Linear(dims[i], dims[i+1])
            nn.init.xavier_normal_(lin.weight); lin.weight.data *= scale
            nn.init.zeros_(lin.bias); net.append(lin)
            if i < len(dims)-2: net.append(nn.Tanh())
        self.net = nn.Sequential(*net)
    def forward(self, x): return self.net(self.ff(x))

def freeze(m, flag):
    for p in m.parameters(): p.requires_grad_(flag)

torch.manual_seed(42)
pinn       = FourierMLP([64,64,64], K=10, scale=1.0).to(device)
source_net = FourierMLP([64,64,64], K=10, scale=0.1).to(device)
print(f'T: {sum(p.numel() for p in pinn.parameters())} params')
print(f'f: {sum(p.numel() for p in source_net.parameters())} params')

In [ ]:
# === Data ===
def f_true(x): return np.sin(np.pi*x) + 0.5*np.sin(3*np.pi*x)
def T_true(x): return np.sin(np.pi*x)/np.pi**2 + 0.5*np.sin(3*np.pi*x)/(9*np.pi**2)

N_OBS=25; NOISE=0.01
x_obs_np = np.linspace(0.05,0.95,N_OBS)
T_clean = T_true(x_obs_np)
T_obs_np = T_clean + NOISE*np.max(np.abs(T_clean))*np.random.randn(N_OBS)

x_obs = torch.tensor(x_obs_np,dtype=torch.float32).unsqueeze(1).to(device)
T_obs = torch.tensor(T_obs_np,dtype=torch.float32).unsqueeze(1).to(device)
N_COL=400
x_col = torch.linspace(0,1,N_COL).unsqueeze(1).to(device)
dx = 1.0/(N_COL-1)
x_eval = torch.linspace(0,1,500).unsqueeze(1).to(device)
x_plot = np.linspace(0,1,500)
x0=torch.zeros(1,1).to(device); x1=torch.ones(1,1).to(device)
print(f'{N_OBS} obs, {NOISE*100:.0f}% noise, {N_COL} collocation')

## Weak-form PDE loss

$$R_k = \int_0^1 T'\phi_k'\,dx - \int_0^1 f\,\phi_k\,dx = 0, \quad \phi_k = \sin(k\pi x)$$

Only needs $T'$ (first derivative) — **no $T''$!**

In [ ]:
# === Weak-form PDE loss ===
K_TEST = 12
k_vals = torch.arange(1,K_TEST+1,dtype=torch.float32).to(device)
xcf = x_col.squeeze()
# phi_k and dphi_k on collocation grid: shape (K, N_COL)
phi  = torch.sin(k_vals[:,None] * np.pi * xcf[None,:])
dphi = k_vals[:,None] * np.pi * torch.cos(k_vals[:,None] * np.pi * xcf[None,:])
# Trapezoidal weights
trap = torch.ones(N_COL,device=device)*dx
trap[0]*=0.5; trap[-1]*=0.5

def weak_loss(dTdx, f_vals):
    """Weak-form PDE: only needs T' (first derivative)."""
    dT = dTdx.squeeze(); f = f_vals.squeeze()
    lhs = (dT[None,:] * dphi * trap[None,:]).sum(1)  # int T'*phi_k' dx
    rhs = (f[None,:]  * phi  * trap[None,:]).sum(1)   # int f*phi_k dx
    return ((lhs - rhs)**2).mean()

def get_dTdx(model, x):
    """First derivative only."""
    x_ = x.detach().clone().requires_grad_(True)
    T = model(x_)
    dT = torch.autograd.grad(T, x_, torch.ones_like(T), create_graph=True)[0]
    return dT

print(f'Weak-form: {K_TEST} test functions sin(k*pi*x). Only needs dT/dx!')

## Phase 1 — Fit $\mathcal{N}_T$ (data + BC, no smoothness penalty)

The weak form only needs $T'$, so no smoothness constraint on $T''$ is needed.

In [ ]:
# === Phase 1: T regression ===
freeze(pinn, True); freeze(source_net, False)
P1 = 20_000
opt1 = optim.Adam(pinn.parameters(), lr=1e-3)
sch1 = optim.lr_scheduler.StepLR(opt1, step_size=8000, gamma=0.3)
h1 = []
print(f'Phase 1: {P1} epochs')
for ep in range(1, P1+1):
    opt1.zero_grad()
    Lbc = pinn(x0).squeeze()**2 + pinn(x1).squeeze()**2
    Ld  = ((pinn(x_obs)-T_obs)**2).mean()
    loss = 500*Lbc + 1000*Ld
    loss.backward(); opt1.step(); sch1.step()
    h1.append({'bc':Lbc.item(),'data':Ld.item()})
    if ep%5000==0:
        print(f'  {ep:6d} | data {Ld.item():.3e} | bc {Lbc.item():.3e}')
        free_memory()
with torch.no_grad(): T_p1 = pinn(x_eval).cpu().squeeze().numpy()
e1 = np.abs(T_p1 - T_true(x_plot))
print(f'Phase 1 done | T L2={np.mean(e1**2)**0.5:.2e}')

In [ ]:
# Phase 1 diagnostic: T' quality (what weak form uses)
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].plot(x_plot, T_true(x_plot), 'b-', lw=2.5, label='$T^*$')
axes[0].plot(x_plot, T_p1, 'r--', lw=2, label='$\\hat{T}$')
axes[0].scatter(x_obs_np, T_obs_np, c='gray', s=40, zorder=5, label='Obs.')
axes[0].set_title(f'T(x) | L2={np.mean(e1**2)**0.5:.2e}'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Check T' (first derivative)
dT_ = get_dTdx(pinn, x_col).detach().cpu().squeeze().numpy()
xc = x_col.cpu().squeeze().numpy()
dT_true = np.cos(np.pi*xc)/np.pi + 0.5*np.cos(3*np.pi*xc)/(3*np.pi)
axes[1].plot(xc, dT_true, 'b-', lw=2.5, label="$T'^*$")
axes[1].plot(xc, dT_, 'r--', lw=1.5, label="$\\hat{T}'$")
axes[1].set_title(f"$T'$ quality | err={np.mean((dT_-dT_true)**2)**0.5:.4f}")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Phase 1: T and FIRST derivative (used by weak form)',fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/phase1.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close(); free_memory()

## Phase 2 — Identify $f$ via weak-form (frozen $\hat{T}$)

Pre-compute $\hat{T}'$ once (frozen). Train $\mathcal{N}_f$ with weak-form + Tikhonov.

In [ ]:
# === Phase 2: f via weak form ===
freeze(pinn, False); freeze(source_net, True)
P2 = 50_000; W2R = 1e-4

# Pre-compute frozen T'
dT_frozen = get_dTdx(pinn, x_col).detach()  # (N_COL,1), no graph
print(f'Phase 2: {P2} epochs | weak-form (no T\'\' needed!)')

opt2 = optim.Adam(source_net.parameters(), lr=1e-3)
sch2 = optim.lr_scheduler.StepLR(opt2, step_size=15000, gamma=0.3)
h2 = []; snaps2 = []

for ep in range(1, P2+1):
    opt2.zero_grad()
    Lw = weak_loss(dT_frozen, source_net(x_col))
    # Tikhonov on f'
    xr = x_col.detach().clone().requires_grad_(True)
    fr = source_net(xr)
    df = torch.autograd.grad(fr, xr, torch.ones_like(fr), create_graph=True)[0]
    Lr = (df**2).mean()
    loss = Lw + W2R*Lr
    loss.backward(); opt2.step(); sch2.step()
    h2.append({'weak':Lw.item(),'reg':Lr.item()})
    if ep%500==0 or ep==1:
        with torch.no_grad(): snaps2.append((ep, source_net(x_eval).cpu().squeeze().numpy().copy()))
    if ep%10000==0:
        print(f'  {ep:6d} | weak {Lw.item():.3e} | reg {Lr.item():.3e}')
        free_memory()

with torch.no_grad(): f_p2 = source_net(x_eval).cpu().squeeze().numpy()
e2 = np.abs(f_p2 - f_true(x_plot))
print(f'Phase 2 done | f L2={np.mean(e2**2)**0.5:.2e}')

In [ ]:
# Phase 2 plot
plt.figure(figsize=(7,4))
plt.plot(x_plot, f_true(x_plot), 'b-', lw=2.5, label='$f^*$ true')
plt.plot(x_plot, f_p2, 'r--', lw=2, label='$\\mathcal{N}_f$ Phase 2')
plt.title(f'Phase 2 (weak-form) | f L2={np.mean(e2**2)**0.5:.2e}')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/phase2.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close(); free_memory()

## Phase 3 — Joint fine-tuning (weak-form, PDE ramp)

PDE weight ramps from 0 to 100 over 30k epochs to preserve the good initialization.

In [ ]:
# === Phase 3: joint fine-tuning ===
freeze(pinn, True); freeze(source_net, True)
P3 = 80_000; RAMP = 30_000
WP = 100.0; WB = 500.0; WD = 100.0; WR = 1e-5

allp = list(pinn.parameters()) + list(source_net.parameters())
opt3 = optim.Adam(allp, lr=3e-4)
sch3 = optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=P3, eta_min=1e-6)
h3 = []; snaps3 = []
print(f'Phase 3: {P3} epochs | PDE ramp 0->{WP} over {RAMP}')

for ep in range(1, P3+1):
    opt3.zero_grad()
    wp = WP * min(1.0, ep/RAMP)
    dT = get_dTdx(pinn, x_col)
    Lw = weak_loss(dT, source_net(x_col))
    Lb = pinn(x0).squeeze()**2 + pinn(x1).squeeze()**2
    Ld = ((pinn(x_obs)-T_obs)**2).mean()
    xr = x_col.detach().clone().requires_grad_(True)
    df = torch.autograd.grad(source_net(xr), xr, torch.ones(N_COL,1,device=device), create_graph=True)[0]
    Lr = (df**2).mean()
    loss = wp*Lw + WB*Lb + WD*Ld + WR*Lr
    loss.backward()
    torch.nn.utils.clip_grad_norm_(allp, 1.0)
    opt3.step(); sch3.step()
    h3.append({'weak':Lw.item(),'bc':Lb.item(),'data':Ld.item(),'reg':Lr.item()})
    if ep%500==0 or ep==1:
        with torch.no_grad():
            snaps3.append((ep, x_eval.cpu().squeeze().numpy().copy(),
                           pinn(x_eval).cpu().squeeze().numpy().copy(),
                           source_net(x_eval).cpu().squeeze().numpy().copy()))
    if ep%10000==0:
        print(f'  {ep:6d} | weak {Lw.item():.3e} | data {Ld.item():.3e} | bc {Lb.item():.3e} | wp {wp:.0f}')
        free_memory()

with torch.no_grad():
    Ta = pinn(x_eval).cpu().squeeze().numpy()
    fa = source_net(x_eval).cpu().squeeze().numpy()
print(f'Adam done | T L2={np.mean((Ta-T_true(x_plot))**2)**0.5:.2e} | f L2={np.mean((fa-f_true(x_plot))**2)**0.5:.2e}')

In [ ]:
# === Phase 3b: L-BFGS polish ===
LB = 500
olb = optim.LBFGS(allp, lr=1.0, max_iter=20, history_size=50, line_search_fn='strong_wolfe')
hlb = []
print(f'L-BFGS: {LB} steps')
for s in range(1, LB+1):
    def closure():
        olb.zero_grad()
        dT = get_dTdx(pinn, x_col)
        Lw = weak_loss(dT, source_net(x_col))
        Lb = pinn(x0).squeeze()**2 + pinn(x1).squeeze()**2
        Ld = ((pinn(x_obs)-T_obs)**2).mean()
        xr = x_col.detach().clone().requires_grad_(True)
        df = torch.autograd.grad(source_net(xr), xr, torch.ones(N_COL,1,device=device), create_graph=True)[0]
        loss = WP*Lw + WB*Lb + WD*Ld + WR*(df**2).mean()
        loss.backward(); return loss
    v = olb.step(closure)
    hlb.append(v.item() if torch.is_tensor(v) else v)
    if s%100==0: print(f'  {s:4d} | {hlb[-1]:.3e}'); free_memory()

with torch.no_grad():
    T_final = pinn(x_eval).cpu().squeeze().numpy()
    f_final = source_net(x_eval).cpu().squeeze().numpy()
snaps3.append(('final', x_eval.cpu().squeeze().numpy(), T_final, f_final))
Te = np.abs(T_final-T_true(x_plot)); fe = np.abs(f_final-f_true(x_plot))
print(f'\nFINAL: T max={Te.max():.2e} L2={np.mean(Te**2)**0.5:.2e}')
print(f'       f max={fe.max():.2e} L2={np.mean(fe**2)**0.5:.2e}')

## Final Results

In [ ]:
# === Final results ===
xn = x_eval.cpu().squeeze().numpy()
fig, axes = plt.subplots(1,2,figsize=(14,5))
ax=axes[0]
ax.plot(x_plot,T_true(x_plot),'b-',lw=2.5,label='Analytical $T^*$')
ax.plot(xn,T_final,'r--',lw=2,label='PINN')
ax.scatter(x_obs_np,T_obs_np,c='gray',s=40,zorder=5,alpha=0.8,label='Obs.')
ax.set_title('Temperature $T(x)$',fontsize=13); ax.set_xlabel('x')
ax.legend(); ax.grid(alpha=0.3)
ax.text(0.05,0.95,f'Max:{Te.max():.2e}\nL2:{np.mean(Te**2)**0.5:.2e}',
        transform=ax.transAxes,va='top',bbox=dict(boxstyle='round',facecolor='wheat',alpha=0.8),fontsize=10)
ax=axes[1]
ax.plot(x_plot,f_true(x_plot),'b-',lw=2.5,label='True $f^*$')
ax.plot(xn,f_final,'r--',lw=2,label='PINN recovered')
ax.set_title('Source $f(x)$ recovered',fontsize=13); ax.set_xlabel('x')
ax.legend(); ax.grid(alpha=0.3)
ax.text(0.05,0.95,f'Max:{fe.max():.2e}\nL2:{np.mean(fe**2)**0.5:.2e}',
        transform=ax.transAxes,va='top',bbox=dict(boxstyle='round',facecolor='lightblue',alpha=0.8),fontsize=10)
plt.suptitle('PINN Inverse - v6 Weak-Form',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/final_results.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close(); free_memory()

In [ ]:
# === Strong-form residual diagnostic ===
x_ = x_col.detach().clone().requires_grad_(True)
T_ = pinn(x_); dT_ = torch.autograd.grad(T_,x_,torch.ones_like(T_),create_graph=True)[0]
d2T_ = torch.autograd.grad(dT_,x_,torch.ones_like(dT_),create_graph=False)[0]
res = (d2T_.detach() + source_net(x_col).detach()).squeeze().cpu().numpy()
xc = x_col.cpu().squeeze().numpy()
fig,ax = plt.subplots(figsize=(7,3))
ax.plot(xc,res,'b-',lw=1.5); ax.axhline(0,color='k',ls=':',lw=0.5)
ax.set_title(f'Strong-form residual | max={np.abs(res).max():.3e} | rms={np.sqrt(np.mean(res**2)):.3e}')
ax.set_xlabel('x'); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/residual.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close()

In [ ]:
# === Loss histories ===
fig,axes = plt.subplots(1,4,figsize=(20,4))
axes[0].semilogy([h['data'] for h in h1],label='Data',alpha=0.8)
axes[0].semilogy([h['bc'] for h in h1],label='BC',alpha=0.6)
axes[0].set_title('Phase 1'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].semilogy([h['weak'] for h in h2],label='Weak PDE')
axes[1].semilogy([h['reg'] for h in h2],label='Tikhonov',ls='--')
axes[1].set_title('Phase 2'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
for k,ls in [('weak','-'),('bc','--'),('data','-.'),('reg',':')]:
    axes[2].semilogy([h[k] for h in h3],ls=ls,label=k.upper(),alpha=0.8)
axes[2].set_title('Phase 3'); axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(alpha=0.3)
axes[3].semilogy(hlb,'b-',lw=1.5)
axes[3].set_title('L-BFGS'); axes[3].set_xlabel('Step'); axes[3].grid(alpha=0.3)
plt.suptitle('Loss History - v6',fontweight='bold'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/loss_history.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close(); free_memory()

In [ ]:
# === Animation ===
fig,axes = plt.subplots(1,2,figsize=(11,4))
def animate(i):
    lbl,xs,Ts,fs = snaps3[i]
    for ax in axes: ax.cla()
    axes[0].plot(x_plot,T_true(x_plot),'b-',lw=2,alpha=0.5,label='$T^*$')
    axes[0].plot(xs,Ts,'r-',lw=2,label='PINN')
    axes[0].scatter(x_obs_np,T_obs_np,c='gray',s=25,zorder=5)
    axes[0].set_ylim(-0.005,0.14); axes[0].set_title(f'T ep {lbl}'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(x_plot,f_true(x_plot),'b-',lw=2,alpha=0.5,label='$f^*$')
    axes[1].plot(xs,fs,'r-',lw=2,label='PINN')
    axes[1].set_ylim(-1.5,1.8); axes[1].set_title(f'f ep {lbl}'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
ani = animation.FuncAnimation(fig,animate,frames=len(snaps3),interval=200)
gp = f'{RESULTS_DIR}/animation.gif'
ani.save(gp,writer='pillow',fps=4,dpi=80); plt.close(); free_memory()
print(f'GIF: {os.path.getsize(gp)//1024} KB')
from IPython.display import Image; Image(gp)

## Noise sensitivity

In [ ]:
# === Noise sensitivity ===
def train_v6(xo, To, e1=10000, e2=20000, e3=30000, lb=200, wr=1e-4):
    torch.manual_seed(0)
    p = FourierMLP([64,64,64],K=10,scale=1.0).to(device)
    s = FourierMLP([64,64,64],K=10,scale=0.1).to(device)
    # P1
    freeze(p,True); freeze(s,False)
    o = optim.Adam(p.parameters(),lr=1e-3)
    for e in range(e1):
        o.zero_grad()
        l = 500*(p(x0).squeeze()**2+p(x1).squeeze()**2)+1000*((p(xo)-To)**2).mean()
        l.backward(); o.step()
    # P2
    freeze(p,False); freeze(s,True)
    dTf = get_dTdx(p, x_col).detach()
    o2 = optim.Adam(s.parameters(),lr=1e-3)
    for e in range(e2):
        o2.zero_grad()
        Lw = weak_loss(dTf, s(x_col))
        xr = x_col.detach().clone().requires_grad_(True)
        df = torch.autograd.grad(s(xr),xr,torch.ones_like(xr),create_graph=True)[0]
        (Lw + wr*(df**2).mean()).backward(); o2.step()
    # P3
    freeze(p,True); freeze(s,True)
    ap = list(p.parameters())+list(s.parameters())
    o3 = optim.Adam(ap,lr=3e-4)
    for e in range(e3):
        o3.zero_grad()
        w = 100*min(1,(e+1)/10000)
        dT = get_dTdx(p, x_col)
        Lw = weak_loss(dT, s(x_col))
        l = w*Lw + 500*(p(x0).squeeze()**2+p(x1).squeeze()**2) + 100*((p(xo)-To)**2).mean()
        l.backward(); torch.nn.utils.clip_grad_norm_(ap,1.0); o3.step()
    ol = optim.LBFGS(ap,lr=1.0,max_iter=20,history_size=50,line_search_fn='strong_wolfe')
    for _ in range(lb):
        def cl():
            ol.zero_grad()
            dT = get_dTdx(p, x_col)
            l = 100*weak_loss(dT,s(x_col))+500*(p(x0).squeeze()**2+p(x1).squeeze()**2)+100*((p(xo)-To)**2).mean()
            l.backward(); return l
        ol.step(cl)
    with torch.no_grad(): fp = s(x_eval).cpu().squeeze().numpy().copy()
    del p,s; free_memory(); return fp

noises = [0.0,0.01,0.02,0.05,0.10]
rn = {}; xn = x_eval.cpu().squeeze().numpy()
print('Noise sensitivity')
for n in noises:
    np.random.seed(0)
    Tn = T_clean + n*np.max(np.abs(T_clean))*np.random.randn(N_OBS)
    Tt = torch.tensor(Tn,dtype=torch.float32).unsqueeze(1).to(device)
    fp = train_v6(x_obs,Tt)
    l2 = float(np.mean((fp-f_true(xn))**2)**0.5)
    rn[n] = (fp,l2); print(f'  {n*100:5.1f}% -> L2={l2:.4f}')

fig,axes = plt.subplots(1,2,figsize=(14,5))
cm = plt.cm.plasma
axes[0].plot(x_plot,f_true(x_plot),'k-',lw=3,label='$f^*$',zorder=10)
for i,(n,(fp,_)) in enumerate(rn.items()):
    axes[0].plot(xn,fp,'--',lw=1.5,color=cm(i/len(noises)),label=f'{n*100:.0f}%')
axes[0].set_title('Recovery vs noise'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot([n*100 for n in noises],[rn[n][1] for n in noises],'ro-',ms=8)
axes[1].set_xlabel('Noise (%)'); axes[1].set_ylabel('L2'); axes[1].grid(alpha=0.3)
axes[1].set_title('L2 vs noise')
plt.suptitle('Noise Sensitivity (v6)',fontweight='bold'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/noise.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close(); free_memory()

In [ ]:
# === Tikhonov study ===
regs = [0.0,1e-5,1e-4,1e-3,1e-2]; rr = {}
np.random.seed(0)
Tn = T_clean+0.03*np.max(np.abs(T_clean))*np.random.randn(N_OBS)
Tt = torch.tensor(Tn,dtype=torch.float32).unsqueeze(1).to(device)
print('Tikhonov (3% noise)')
for w in regs:
    fp = train_v6(x_obs,Tt,wr=w)
    l2 = float(np.mean((fp-f_true(xn))**2)**0.5)
    rr[w] = (fp,l2); print(f'  {w:.0e} -> L2={l2:.4f}')
plt.figure(figsize=(9,5))
plt.plot(x_plot,f_true(x_plot),'k-',lw=3,label='$f^*$')
cm2 = plt.cm.viridis
for i,(w,(fp,l2)) in enumerate(rr.items()):
    plt.plot(xn,fp,'--',lw=1.8,color=cm2(i/len(regs)),label=f'$w_{{reg}}={w:.0e}$ (L2={l2:.3f})')
plt.title('Tikhonov (noise=3%)',fontsize=12); plt.xlabel('x')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/tikhonov.png',dpi=120,bbox_inches='tight')
plt.show(); plt.close(); free_memory()

In [ ]:
# === Summary ===
import glob
print('='*55)
print('  v6 RESULTS (weak-form PDE)')
print('='*55)
print(f'  T L2: {np.mean(Te**2)**0.5:.2e}')
print(f'  f L2: {np.mean(fe**2)**0.5:.2e}')
print(f'  Obs: {N_OBS}, noise={NOISE*100:.0f}%')
print(f'  Arch: FourierMLP(K=10,[64,64,64])')
print(f'  Phases: P1={P1} P2={P2} P3={P3}+{LB}')
print(f'  Key: weak-form (no T\'\')')
print('='*55)
for fp in sorted(glob.glob(f'{RESULTS_DIR}/*')):
    print(f'  {os.path.basename(fp):35s} {os.path.getsize(fp)//1024:>5} KB')